# What's new in `unify-compilation-workflow`

A chipper tour of everything new on this branch — the unified
compilation workflow that turns AIE designs into Triton-style JIT
functions, packaged kernel factories, content-addressed caching,
tensor-arg validation, runtime/trace tooling, and the bench/verify
helpers that go with them.

Every cell here runs on a Phoenix NPU with `ironenv` activated.

**TL;DR — what's new:**

1. `@iron.jit` decorator — one Python function = one JIT'd design.
2. `aie.iron.kernels.*` factories — packaged C++ kernel wrappers (no more hand-rolled `Kernel(...)` + Makefile rules).
3. `Compile[T]` / `In` / `Out` / `InOut` annotation markers + 3 ways to bind compile params.
4. `CompilableDesign` / `CallableDesign` API: `lower()` / `specialize()` / `compile()` / `__call__`.
5. Content-addressed kernel cache in `~/.npu/cache/`, with hash invalidation on Peano upgrade *and* Peano-absent tolerance.
6. Tensor-arg validation with multi-DMA fan-out support.
7. `aie.utils.benchmark` (`run_iters`, `print_benchmark`).
8. `aie.utils.verify` (`nearly_equal`, `count_mismatches`).
9. `aie.utils.trace` integration: TraceConfig + JIT + `trace_to_json` + `print_cycles_summary`.
10. Compile-process logging: every clang/aiecc command goes through `logging.getLogger("aie.utils.compile")`.

## 0. Setup

Pick a device and import the iron surface.

In [1]:
import logging
import time
from pathlib import Path

import numpy as np

import aie.iron as iron
from aie.iron import (
    Compile,
    In,
    Out,
    ObjectFifo,
    Program,
    Runtime,
    Worker,
    jit,
    kernels,
)
from aie.iron.controlflow import range_

# No need to call set_current_device — iron.get_current_device() falls back
# to DefaultNPURuntime.device(), which queries XRT for the actual hardware
# (NPU1 = Phoenix, NPU2 = Strix). Both `lower()` and on-NPU calls use it.
device = iron.get_current_device()
print("Auto-detected device:", type(device).__name__)

Auto-detected device: NPU1


## 1. `@iron.jit` — Triton-style design compilation

The decorator turns a Python function that *generates* an MLIR module
into something you call like a normal function. The body runs at
*compile* time; the returned `Program(...).resolve_program()` becomes
the cached kernel.

A minimal passthrough fits in 20 lines:

In [2]:
@jit
def passthrough(x: In, y: Out, *, N: Compile[int]):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]

    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `passthrough` is now a CallableDesign, not a function.
print(repr(passthrough))

CallableDesign(CompilableDesign(generator='passthrough', compile_kwargs={}))


### Lower it (cheap) — generate MLIR without the slow aiecc step

`lower()` runs the generator and returns the MLIR module as a string.
Useful for sanity-checking the structure before the full compile.

In [3]:
mlir = passthrough.lower(N=4096)
print("\n".join(mlir.splitlines()[:6]))
print(f"... ({len(mlir.splitlines())} lines total)")

module {
  aie.device(npu1) {
    %logical_core = aie.logical_tile<CoreTile>(?, ?)
    %logical_shim_noc = aie.logical_tile<ShimNOCTile>(?, ?)
    %logical_shim_noc_0 = aie.logical_tile<ShimNOCTile>(?, ?)
    aie.objectfifo @in(%logical_shim_noc, {%logical_core}, 2 : i32) : !aie.objectfifo<memref<4096xui8>> 
... (44 lines total)


### Run it (compiles to xclbin on first call, then cached)

Calling the design with `iron.tensor` inputs JIT-compiles
(if not already cached) and runs on the NPU.

In [4]:
N = 4096
in_t = iron.arange(N, dtype=np.uint8, device="npu")
out_t = iron.zeros(N, dtype=np.uint8, device="npu")

passthrough(in_t, out_t, N=N)
assert np.array_equal(in_t.numpy(), out_t.numpy())
print(f"PASS — {N} bytes round-tripped through the NPU.")

PASS — 4096 bytes round-tripped through the NPU.


## 2. The kernel library: `aie.iron.kernels`

Before this branch, every design hand-rolled `Kernel("conv2dk1_i8",
"conv2dk1_i8.o", [int8_input_ty, ...])` declarations *and* maintained
a Makefile rule that compiled the `.cc` to `.o` with the right
`-D{INT8,UINT8}_ACT` flag.

The kernel library packages all of that — source path, compile flags,
typed argument list — behind one factory call.

In [5]:
conv = kernels.conv2dk1(
    input_width=32,
    input_channels=64,
    output_channels=64,
    act_dtype=np.int8,
)
# `arg_types` is public.  The other introspection attributes
# (`_name`, `_source_file`, `_compile_flags`) are still private — we peek
# at them here purely to show what the factory packaged for you.
_src = conv._source_file
# Show the repo-relative path. The install layout mirrors the repo from
# `aie_kernels/...` onwards, so strip everything up to that prefix.
_repo_rel = "aie_kernels/" + _src.partition("aie_kernels/")[2]
print(f"function name : {conv._name}")
print(f"source        : {_repo_rel}")
print(f"compile flags : {conv._compile_flags}")
print(f"arg count     : {len(conv.arg_types())}")
for i, t in enumerate(conv.arg_types()[:3]):
    print(f"  arg[{i}] : {t}")
print("  ...")

function name : conv2dk1_i8
source        : aie_kernels/aie2/conv2dk1.cc
compile flags : ['-DINT8_ACT']
arg count     : 7
  arg[0] : numpy.ndarray[(2048,), numpy.dtype[numpy.int8]]
  arg[1] : numpy.ndarray[(4096,), numpy.dtype[numpy.int8]]
  arg[2] : numpy.ndarray[(2048,), numpy.dtype[numpy.uint8]]
  ...


The library covers conv variants, eltwise, reductions, activations
(ReLU, GeLU, SwiGLU, softmax), vision (filter2d, RGBA conversions),
matmul (`mm`, `mm_zero`, `cascade_mm`, `mv`), LUT-backed `bf16_exp`,
and bottleneck conv combos:

In [6]:
import aie.iron.kernels as K

named = sorted(n for n in dir(K) if not n.startswith("_") and not n[0].isupper())
factories = [
    n
    for n in named
    if n not in ("conv", "eltwise", "linalg", "reduce", "vision", "activation")
]
print(f"{len(factories)} factories available:")
for i in range(0, len(factories), 6):
    print("  " + ", ".join(factories[i : i + 6]))

36 factories available:
  add, add_weighted, bf16_exp, bitwise_and, bitwise_or, bn_conv2dk1_i8
  bn_conv2dk1_relu, bn_conv2dk1_skip, bn_conv2dk3, bn_conv2dk3_dw, cascade_mm, conv2dk1
  conv2dk14, conv2dk1_i8, conv2dk1_skip, conv2dk1_skip_init, conv2dk3, filter2d
  gelu, gray2rgba, mm, mm_zero, mul, mv
  passthrough, reduce_add, reduce_max, reduce_min, relu, rgba2gray
  rgba2hue, scale, silu, softmax, swiglu, threshold


### New on this branch: shared-buffer parameters

Two factories grew optional knobs for designs that share weight buffers
across multiple workers — both motivated by the resnet conv2_x port.

- **`kernels.conv2dk1_skip_init(skip_input_channels=...)`** — sizes the
  weights buffer for inline skip-projection weights and the skip buffer
  for the unprojected channel count.
- **`kernels.conv2dk3(weight_output_channels=...)`** — sizes the weights
  buffer for *more* output channels than a single call produces; multiple
  workers share one buffer and pick their slice via `channel_offset`.

In [7]:
def wt_skip(k):
    return k.arg_types()[2].__args__[0][0], k.arg_types()[4].__args__[0][0]


# Default: skip_input_channels falls back to input_channels.
default = kernels.conv2dk1_skip_init(input_channels=64, output_channels=256)
print(
    f"default (skip_in=64)   : wt={wt_skip(default)[0]:>6} bytes, "
    f"skip={wt_skip(default)[1]:>5} bytes"
)

# Explicit: skip path has narrower input (e.g. a lower-res branch).
narrow = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=32,
)
print(
    f"narrow  (skip_in=32)   : wt={wt_skip(narrow)[0]:>6} bytes, "
    f"skip={wt_skip(narrow)[1]:>5} bytes"
)

# Explicit: skip path is wider.
wide = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=128,
)
print(
    f"wide    (skip_in=128)  : wt={wt_skip(wide)[0]:>6} bytes, "
    f"skip={wt_skip(wide)[1]:>5} bytes"
)

print("\n  weights = (input_channels + skip_input_channels) * output_channels")
print("  skip    = input_width * skip_input_channels")

default (skip_in=64)   : wt= 32768 bytes, skip= 2048 bytes
narrow  (skip_in=32)   : wt= 24576 bytes, skip= 1024 bytes
wide    (skip_in=128)  : wt= 49152 bytes, skip= 4096 bytes

  weights = (input_channels + skip_input_channels) * output_channels
  skip    = input_width * skip_input_channels


In [8]:
default = kernels.conv2dk3(input_channels=64, output_channels=64)
shared = kernels.conv2dk3(
    input_channels=64,
    output_channels=32,
    weight_output_channels=64,
)


def wt_out(k):
    return k.arg_types()[3].__args__[0][0], k.arg_types()[4].__args__[0][0]


print(
    f"conv2dk3 defaults      : wt={wt_out(default)[0]:>5}, out={wt_out(default)[1]:>5}"
)
print(f"conv2dk3 shared-buffer : wt={wt_out(shared)[0]:>5}, out={wt_out(shared)[1]:>5}")
print("  (two workers share the 36864-byte weights, each writes 1024 bytes)")

conv2dk3 defaults      : wt=36864, out= 2048
conv2dk3 shared-buffer : wt=36864, out= 1024
  (two workers share the 36864-byte weights, each writes 1024 bytes)


## 3. `Compile[T]` parameters: three flavours

Compile-time parameters use `Compile[type]` and **must be keyword-only**
(place a `*,` before them). Three ways to bind them, mirroring Triton:

In [9]:
@jit  # A: bare decorator, params at call time
def gemm_a(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit(M=512, N=512)  # B: pre-bound params at decoration
def gemm_b(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit  # C: defaults in signature
def gemm_c(a: In, b: In, c: Out, *, M: Compile[int] = 256, N: Compile[int] = 256):
    pass


print("a:", gemm_a.compilable.compile_kwargs)
print("b:", gemm_b.compilable.compile_kwargs)
print("c:", gemm_c.compilable.compile_kwargs, "(uses signature defaults at lower-time)")

a: {}
b: {'M': 512, 'N': 512}
c: {} (uses signature defaults at lower-time)


Unknown kwargs to `@iron.jit` raise `TypeError` immediately at decoration
time — fails fast so typos don't silently run a kernel with no value bound.
Non-keyword-only `Compile[T]` params raise `TypeError` with a fix-it
suggestion.

In [10]:
try:

    @jit(NOPE=99)
    def oops(x: In, y: Out, *, N: Compile[int]):
        pass

except TypeError as e:
    print("TypeError:", e)

TypeError: @iron.jit received keyword argument(s) that do not match any Compile[T]-annotated parameter of 'oops': ['NOPE'].
  Valid Compile[T] params: ['N'].
  Config keys: ['aiecc_flags', 'compile_flags', 'include_paths', 'object_files', 'source_files', 'trace_config', 'use_cache'].


## 4. `lower()` vs `specialize()` vs `compile()` vs `__call__`

A `CallableDesign` exposes four entry points with very different costs.

| call            | does                                          | cost      |
|-----------------|-----------------------------------------------|-----------|
| `.specialize(**kw)` | bind compile params, return new CallableDesign | µs        |
| `.lower(**kw)`  | run generator → MLIR string                   | ms        |
| `.compile()`    | full aiecc → cached `final.xclbin` + `insts.bin` | s (then cached) |
| `design(*tensors, **kw)` | compile (cached) + run on NPU       | ms after first |


In [11]:
t = time.perf_counter()
spec = passthrough.specialize(N=4096)
print(f"specialize()  : {(time.perf_counter() - t) * 1e6:6.0f} µs")

t = time.perf_counter()
mlir = spec.lower()
print(
    f"lower()       : {(time.perf_counter() - t) * 1e3:6.1f} ms  ({len(mlir):,} chars)"
)

t = time.perf_counter()
xclbin, insts = spec.compile()
print(f"compile()     : {(time.perf_counter() - t) * 1e3:6.1f} ms  (cache hit)")
print(f"  xclbin -> {xclbin}")
print(f"  insts  -> {insts}")

specialize()  :     48 µs
lower()       :   11.1 ms  (2,230 chars)
compile()     :    9.9 ms  (cache hit)
  xclbin -> /home/ehunhoff/.npu/cache/cbd39928bee1447a5782e4e0/final.xclbin
  insts  -> /home/ehunhoff/.npu/cache/cbd39928bee1447a5782e4e0/insts.bin


### Progressive binding: chained `.specialize()`

`specialize()` is **immutable-style** — it returns a *new* `CallableDesign`
with the kwargs merged in, leaving the original unchanged. That makes it
trivial to bind compile params progressively: pin one now, pin the rest
later, override anything along the way. Standard `functools.partial` also
works on `.lower` / `.compile` / `.__call__` since they're plain methods —
no JIT machinery in the way.


In [12]:
import functools


@jit
def add_const(x: In, y: Out, *, N: Compile[int], dtype: Compile[type]):
    line_ty = np.ndarray[(N,), np.dtype[dtype]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i] + 1
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# Chain: bind N now, bind dtype later. Each call returns a fresh design.
half = add_const.specialize(N=4096)
full = half.specialize(dtype=np.int32)
print("half:    ", repr(half))
print("full:    ", repr(full))
print("original:", repr(add_const), "  # untouched")

# Override: re-specialize the same key.
wider = full.specialize(N=8192)
print("wider:   ", repr(wider))

# functools.partial works too — .lower is just a method.
lower_N4096 = functools.partial(add_const.lower, N=4096)
mlir = lower_N4096(dtype=np.int32)
print(f"\nfunctools.partial(.lower, N=4096)(dtype=int32) -> {len(mlir):,} chars MLIR")

half:     CallableDesign(CompilableDesign(generator='add_const', compile_kwargs={'N': 4096}))
full:     CallableDesign(CompilableDesign(generator='add_const', compile_kwargs={'N': 4096, 'dtype': <class 'numpy.int32'>}))
original: CallableDesign(CompilableDesign(generator='add_const', compile_kwargs={}))   # untouched
wider:    CallableDesign(CompilableDesign(generator='add_const', compile_kwargs={'N': 8192, 'dtype': <class 'numpy.int32'>}))

functools.partial(.lower, N=4096)(dtype=int32) -> 2,317 chars MLIR


### Peek at what `compile()` actually runs

All compile-path subprocesses (`clang++` for ExternalFunctions, `aiecc`
for the design itself) go through `logging.getLogger("aie.utils.compile")`.
Bumping it to `DEBUG` reveals every command line plus cache-hit/miss
messages — enough to grab the exact `clang++` invocation aiecc handed
off, paste it into a terminal, and iterate. Force a fresh compile with
`use_cache=False` so you can see every clang/aiecc invocation light up:

In [13]:
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s", force=True)
logging.getLogger("aie.utils.compile").setLevel(logging.DEBUG)


# `use_cache=False` forces a fresh compile; using kernels.passthrough means
# you also see the per-ExternalFunction `clang++` invocation, not just the
# aiecc subprocess. Both lines come from the same logger.
@jit(use_cache=False)
def passthrough_logged(x: In, y: Out, *, N: Compile[int]):
    line_size = N // 4
    line_ty = np.ndarray[(line_size,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")
    pt = kernels.passthrough(tile_size=line_size, dtype=np.uint8)

    def core_fn(of_in, of_out, pt):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        pt(elem_in, elem_out, line_size)
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod(), pt])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


_ = passthrough_logged.specialize(N=4096).compile()

# Quiet the logger again so the rest of the walkthrough isn't noisy.
logging.getLogger("aie.utils.compile").setLevel(logging.WARNING)

aie.utils.compile.jit.compilabledesign | Cache miss for 'passthrough_logged' (hash=e99a4149fad6bc8c9675f798); compiling...


aie.utils.compile.utils | Compiling with: /scratch/ehunhoff/mlir-aie/ironenv/lib/python3.10/site-packages/llvm-aie/bin/clang++ /home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/passThroughLine.cc -c -o /home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/passThroughLine.o -I/scratch/ehunhoff/mlir-aie/install/include -std=c++20 -Wno-parentheses -Wno-attributes -Wno-macro-redefined -Wno-empty-body -O2 -DNDEBUG --target=aie2-none-unknown-elf -I /scratch/ehunhoff/mlir-aie/install/include -I /scratch/ehunhoff/mlir-aie/install/include/aie_kernels/generic -DBIT_WIDTH=8


aie.utils.compile.utils | Running: /scratch/ehunhoff/mlir-aie/install/bin/aiecc /home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/aie.mlir --no-compile-host --no-xchesscc --no-xbridge --peano=/scratch/ehunhoff/mlir-aie/ironenv/lib/python3.10/site-packages/llvm-aie --aie-generate-npu-insts --npu-insts-name=/home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/insts.bin --aie-generate-xclbin --xclbin-name=/home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/final.xclbin --tmpdir=/home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798


aie.utils.compile.utils | 

****** Bootgen v2023.2
  **** Build date : May  9 2026-15:28:35
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.


[INFO]   : Bootimage generated successfully

XRT Build Version: 2.19.0 (HEAD)
       Build Date: 2025-01-14 09:00:07
          Hash ID: acc144998d650acbfda7e5919a1290de8f8c7735
Creating a default 'in-memory' xclbin image.

Section: 'MEM_TOPOLOGY'(6) was successfully added.
Size   : 88 bytes
Format : JSON
File   : '/home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/main_mem_topology.json'

Section: 'AIE_PARTITION'(32) was successfully added.
Size   : 2536 bytes
Format : JSON
File   : '/home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/main_aie_partition.json'
Info: Embedded Metadata section is missing project.platform.device.core element, adding it.
Successfully wrote (8568 bytes) to the output file: /home/ehunhoff/.npu/cache/e99a4149fad6bc8c9675f798/final.xc

## 5. Content-addressed caching

Every compiled design lands in `$NPU_CACHE_HOME/<hash>/` (default
`~/.npu/cache/<hash>/` if `NPU_CACHE_HOME` is unset). The hash is
content-addressed on:

- the generator function's bytecode + closure
- the resolved `compile_kwargs` (sorted, JSON-stable)
- the target arch (`aie2` / `aie2p`)
- Peano's `clang++` mtime (so a Peano upgrade auto-invalidates)
- aiecc's mtime
- any `source_files` mtimes (so a `.cc` edit auto-invalidates)

Re-running an unchanged design is ~free; changing *anything* triggers
a fresh compile.

Each per-design directory keeps **every artifact** aiecc produced —
useful when you need to inspect the lowered MLIR, the post-placement IR,
the per-core LLVM IR, or the actual `.o` files that linked into the
final xclbin:

In [14]:
from aie.utils.compile import NPU_CACHE_HOME

# What each artifact in the per-design cache dir is. (`xx` matches whichever
# row/col the AIE core landed on, e.g. `main_core_0_2.elf`.)
_ARTIFACT_DESC = {
    ".lock": "cross-process flock so concurrent compiles don't collide",
    "aie.mlir": "input MLIR your generator produced (post-resolve_program)",
    "input_with_addresses.mlir": "lowered MLIR with shim/memtile DMA addresses bound",
    "final.xclbin": "the binary XRT loads onto the NPU",
    "insts.bin": "NPU runtime instruction stream (paired with the xclbin)",
    "main.pdi": "Programmable Device Image — config data packed by bootgen",
    "main_aie_partition.json": "AIE partition manifest (which tiles, kernels, memory)",
    "main_aie_cdo_elfs.bin": "CDO blob carrying the per-core ELFs to the device",
    "main_aie_cdo_enable.bin": "CDO blob enabling tiles after init",
    "main_aie_cdo_init.bin": "CDO blob initialising tile/DMA registers",
    "main_design.bif": "Bootgen Image Format spec used to assemble main.pdi",
    "main_kernels.json": "kernel metadata that XRT consumes alongside the xclbin",
    "main_mem_topology.json": "memory-topology descriptor embedded into the xclbin",
    "main_core_xx.elf": "per-core final ELF the AIE tile actually runs",
    "main_core_xx.ld.script": "linker script Peano used to lay out that ELF",
    "main_core_xx.ll": "per-core LLVM IR before opt",
    "main_core_xx.opt.ll": "per-core LLVM IR after opt",
    "main_core_xx.peanohack.ll": "Peano workaround IR (vector-intrinsic fixups)",
    "main_core_xx.o": "per-core compiled object that links into the .elf",
    "passThroughLine.cc": "ExternalFunction kernel source, copied in for clang",
    "passThroughLine.o": "clang's compiled .o for that ExternalFunction",
}


def _describe(name: str) -> str:
    """Look up an artifact, normalising per-core filenames to `xx`."""
    if name in _ARTIFACT_DESC:
        return _ARTIFACT_DESC[name]
    import re

    return _ARTIFACT_DESC.get(re.sub(r"_\d+_\d+", "_xx", name), "")


print(f"Cache root: {NPU_CACHE_HOME}")
print(f"Cache entries on this machine: {len(list(NPU_CACHE_HOME.glob('*')))}")
kernel_dir = Path(xclbin).parent
print(f"\npassthrough(N=4096) lives at: {kernel_dir}\n")
for p in sorted(kernel_dir.iterdir()):
    print(f"  {p.name:<30} {_describe(p.name)}")

Cache root: /home/ehunhoff/.npu/cache
Cache entries on this machine: 131

passthrough(N=4096) lives at: /home/ehunhoff/.npu/cache/cbd39928bee1447a5782e4e0

  .lock                          cross-process flock so concurrent compiles don't collide
  aie.mlir                       input MLIR your generator produced (post-resolve_program)
  final.xclbin                   the binary XRT loads onto the NPU
  input_with_addresses.mlir      lowered MLIR with shim/memtile DMA addresses bound
  insts.bin                      NPU runtime instruction stream (paired with the xclbin)
  main.pdi                       Programmable Device Image — config data packed by bootgen
  main_aie_cdo_elfs.bin          CDO blob carrying the per-core ELFs to the device
  main_aie_cdo_enable.bin        CDO blob enabling tiles after init
  main_aie_cdo_init.bin          CDO blob initialising tile/DMA registers
  main_aie_partition.json        AIE partition manifest (which tiles, kernels, memory)
  main_core_0_2.elf 

## 6. Tensor-arg validation

When you call a JIT design, the framework reads the host-facing
`aie.runtime_sequence`'s typed arguments straight from the lowered MLIR
and checks each runtime tensor's element count against the declared
memref size. The runtime_sequence signature *is* the host-side contract,
so there's no need to walk DMA ops or fold multi-DMA patterns back
together — fan-outs, repeated loads, InOut fill+drain pairs all just
work because the arg type doesn't change.

Modules with multiple runtime_sequences (helper sub-sequences invoked
via `aiex.run`, or per-device sequences in a multi-device program) are
handled by picking the unique **call-graph root** — the sequence no
other sequence calls. If there's no unique root (multi-device with
several roots, or an unnamed sequence we can't reason about) validation
is skipped rather than guessing wrong.

Validation runs on both fresh compiles and cache hits, so wrong-size
tensors fail loudly *before* the NPU runs:

In [15]:
wrong_size = iron.zeros(99, dtype=np.uint8)
try:
    passthrough(wrong_size, out_t, N=4096)
except RuntimeError as e:
    print("Caught expected error:")
    print(f"  {e}")

Caught expected error:
  Tensor argument 'x' has 99 elements but the kernel was compiled for 4096 elements.
Compile[T] parameters used at compile time: {'N': 4096}


In [16]:
# `parse_dma_sizes` is a private helper exposed here just so we can peek
# at what validation expects. After a `compile()` (or any call), the same
# list is also stored on the design at `compilable._expected_tensor_sizes`.
# It reads the entry-point runtime_sequence's typed memref args directly,
# so this matches the host-side contract aiecc generated for the design.
from aie.utils.compile.jit._dma_size_parser import parse_dma_sizes
from aie.iron import In, Out, InOut
import inspect

cache_dir = Path(xclbin).parent
sizes = parse_dma_sizes(cache_dir)

# Direction (In / Out / InOut) lives on the @iron.jit generator's
# signature, not in the runtime_sequence MLIR — re-introspect the original
# generator to pair each arg's element count with its declared kind.
_KIND = {In: "In", Out: "Out", InOut: "InOut"}
sig = inspect.signature(passthrough.compilable.mlir_generator)
tensor_params = [
    (name, _KIND[p.annotation])
    for name, p in sig.parameters.items()
    if p.annotation in _KIND
]

print(f"runtime_sequence arg sizes for passthrough(N=4096): {sizes}")
for i, (size, (name, kind)) in enumerate(zip(sizes, tensor_params)):
    print(f"  arg[{i}] {name!r:8} {kind:5} : {size} elements")

runtime_sequence arg sizes for passthrough(N=4096): [4096, 4096]
  arg[0] 'x'      In    : 4096 elements
  arg[1] 'y'      Out   : 4096 elements


## 7. Helper utilities cheat-sheet

A round-up of the helpers that landed on this branch — what to import
and when to reach for them.

| Helper | What it does |
|--------|--------------|
| `aie.iron.tensor / zeros / ones / rand / randint / arange` | Build host tensors targeting `device="npu"` (or `"cpu"`). |
| `aie.iron.{In, Out, InOut, Compile}` | Type annotation markers for `@jit` generator params. |
| `aie.iron.{jit, CompilableDesign, CallableDesign}` | The JIT decorator + the two design wrapper classes. |
| `aie.iron.kernels.*` | Pre-packaged kernel factories — see §2. |
| `aie.iron.{compile_context, get_compile_arg}` | Inject compile-time values into generators that aren't kwargs of the top-level function (advanced). |
| `aie.utils.benchmark.{run_iters, print_benchmark}` | Warmup + timed iters → stats. |
| `aie.utils.verify.{nearly_equal, count_mismatches}` | Sloppy-equality compare for LUT/saturating kernel outputs. |
| `aie.utils.trace.TraceConfig` | Hardware trace plumbing — see §8. |
| `aie.utils.trace.utils.print_cycles_summary` | Pretty-print per-event cycle counts from a parsed trace. |
| `aie.utils.compile.NPU_CACHE_HOME` | Cache root path (default `~/.npu/cache`; override with `$NPU_CACHE_HOME`). |
| `aie.utils.compile.jit._dma_size_parser.parse_dma_sizes` | Per-host-arg element counts read from the entry-point `aie.runtime_sequence`'s typed args. |
| `aie.utils.set_current_device` | Set the device used by `iron.get_current_device()` and JIT compile. |
| `aie.utils.DefaultNPURuntime` | Cached XRT runtime — auto-picks NPU1/NPU2 from the device's product name. |

**Need more on any of these?** Every helper above has a docstring.
Three ways to read it without leaving the notebook:

* `help(some_helper)` — Python built-in, prints signature + docstring.
* `some_helper?` — Jupyter / IPython shortcut, opens a side panel.
* `print(some_helper.__doc__)` — raw access, works everywhere.

Worked example:

In [17]:
from aie.utils.verify import count_mismatches

help(count_mismatches)

Help on function count_mismatches in module aie.utils.verify:

count_mismatches(actual, ref, *, rtol: 'float' = 0.128, atol: 'float | None' = None, stop_at_nonfinite: 'bool' = True) -> 'tuple[int, int]'
    Count tolerance violations between ``actual`` and ``ref``.
    
    Returns ``(errors, n_checked)`` where ``n_checked`` is the number of
    samples that were actually compared (less than ``len(ref)`` when
    ``stop_at_nonfinite`` halts on the first inf/nan from either side).
    
    With ``stop_at_nonfinite=True`` (default), this matches the canonical
    C++ verify pattern that ``break``\s on the first inf/nan rather than
    treating the LUT's behaviour outside its defined input range as part
    of the contract.



In [18]:
# `compile_context` — inject compile args into nested helpers
from aie.iron import compile_context, get_compile_arg


def make_fifo_pair(line_ty):
    # This helper doesn't take a `name_prefix` arg, but it can read one
    # from the active CompileContext.
    prefix = get_compile_arg("prefix", default="")
    return ObjectFifo(line_ty, name=f"{prefix}in"), ObjectFifo(
        line_ty, name=f"{prefix}out"
    )


with compile_context(prefix="layer1_"):
    line_ty = np.ndarray[(64,), np.dtype[np.uint8]]
    of_in, of_out = make_fifo_pair(line_ty)
    print("Inside context:", of_in.name, of_out.name)

# Outside the context, the default kicks in.
of_in, of_out = make_fifo_pair(line_ty)
print("Outside context:", of_in.name, of_out.name)

Inside context: layer1_in layer1_out
Outside context: in out


## 8. Hardware trace via `TraceConfig`

The hardware trace machinery threads through `@iron.jit` in two steps:

1. **Generator side**: declare `trace_config: Compile[TraceConfig | None] = None`,
   wrap your `Worker(...)` with `trace=1 if trace_config else 0`,
   and call `rt.enable_trace(trace_config.trace_size, workers=[...])`
   inside the runtime sequence.

2. **Caller side**: pass `trace_config=TraceConfig(trace_size=N)` at
   call time. After the call, `trace_config.physical_mlir_path` is
   populated so you can run `trace_config.trace_to_json(...)` to parse
   `trace.txt` into a Chrome-tracing JSON.

This mirrors what `programming_examples/basic/passthrough_kernel/passthrough_kernel.py`
does end-to-end.

In [19]:
from aie.utils.trace import TraceConfig


@jit
def passthrough_with_trace(
    x: In,
    y: Out,
    *,
    N: Compile[int],
    trace_config: Compile[TraceConfig | None] = None,
):
    line_size = N // 4
    tensor_ty = np.ndarray[(N,), np.dtype[np.uint8]]  # what the host hands in
    line_ty = np.ndarray[(line_size,), np.dtype[np.uint8]]  # per-tile chunks
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    # Use the kernel-library passthrough — its C++ body calls event0() at
    # entry and event1() at exit, which is what `print_cycles_summary`
    # below pairs up to count per-kernel cycles. An inline Python copy loop
    # would still produce a real trace but with no event0/event1 pairs, so
    # the summary would report 0 kernel invocations.
    pt = kernels.passthrough(tile_size=line_size, dtype=np.uint8)

    def core_fn(of_in, of_out, pt):
        for _ in range_(4):  # 4 line tiles per N-element invocation
            elem_in = of_in.acquire(1)
            elem_out = of_out.acquire(1)
            pt(elem_in, elem_out, line_size)
            of_in.release(1)
            of_out.release(1)

    # Wire trace=1 onto each worker we want to instrument.
    worker = Worker(
        core_fn, [of_in.cons(), of_out.prod(), pt], trace=1 if trace_config else 0
    )

    rt = Runtime()
    with rt.sequence(tensor_ty, tensor_ty) as (a_in, b_out):
        if trace_config:
            rt.enable_trace(trace_config.trace_size, workers=[worker])
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `lower()` runs the generator and shows the MLIR — confirm the trace
# wiring (`packetflow`s into the trace ddr buffer) was actually emitted.
trace_cfg = TraceConfig(trace_size=8192)
mlir_with_trace = passthrough_with_trace.lower(N=4096, trace_config=trace_cfg)
trace_lines = [
    l for l in mlir_with_trace.splitlines() if "trace" in l.lower() or "packetflow" in l
]
print(f"{len(trace_lines)} trace-related ops in the lowered MLIR:")
for line in trace_lines[:6]:
    print("  " + line.strip()[:100])

17 trace-related ops in the lowered MLIR:
  aie.trace @trace_core_1(%logical_core) {
  aie.trace.mode "Event-Time"
  aie.trace.packet id = 1 type = core
  aie.trace.event <"INSTR_EVENT_0">
  aie.trace.event <"INSTR_EVENT_1">
  aie.trace.event <"INSTR_VECTOR">


**Calling it on hardware** writes a `trace.txt` next to your cwd.
`trace_config.physical_mlir_path` is auto-populated after the call so
`trace_config.trace_to_json(physical_mlir_path, output_name)` parses the
raw trace into a Chrome-tracing JSON, and
`aie.utils.trace.utils.print_cycles_summary` gives you the per-event
cycle breakdown — useful for spotting which kernel call is the
bottleneck. The runnable end-to-end driver also lives in
`programming_examples/basic/passthrough_kernel/passthrough_kernel.py`
(or `make trace`). Inline demo using the design from the cell above:

In [20]:
import os
import json
from aie.utils.trace.utils import print_cycles_summary

trace_cfg = TraceConfig(trace_size=8192)
in_t = iron.arange(N, dtype=np.uint8, device="npu")
out_t = iron.zeros(N, dtype=np.uint8, device="npu")

# Hardware run with trace=on. trace.txt lands in cwd; physical_mlir_path
# gets populated on the cfg so we can parse against the lowered IR.
passthrough_with_trace(in_t, out_t, N=N, trace_config=trace_cfg)
print(f"trace.txt        : {os.path.getsize('trace.txt')} bytes")
# `print(trace_cfg)` uses TraceConfig.__str__ — eval-faithful repr first,
# then any post-run state (physical_mlir_path, last_tensor_*) indented
# underneath. `repr(trace_cfg)` would just be the constructor-args line.
print(trace_cfg)

# Parse to Chrome-tracing JSON (open in chrome://tracing or perfetto).
trace_cfg.trace_to_json(trace_cfg.physical_mlir_path, "trace_demo.json")
events = json.load(open("trace_demo.json"))
print(f"trace_demo.json  : {len(events)} events")

# print_cycles_summary walks the JSON and reports per-tile / per-event
# cycle counts. For this single-iter passthrough it just confirms the
# trace landed on the expected core; richer designs surface kernel-level
# cycle hot-spots here.
print()
print_cycles_summary("trace_demo.json")

trace.txt        : 575 bytes
TraceConfig(trace_size=8192)
  physical_mlir_path=/home/ehunhoff/.npu/cache/bc932c6806ac16c852f37931/input_with_addresses.mlir
trace_demo.json  : 76 events

core_trace for tile2,1
Total number of full kernel invocations is 4
First/Min/Avg/Max cycles is 45/ 45/ 45.0/ 45


## 9. `aie.utils.benchmark` — timing + warmup

`run_iters(fn, *args, warmup=N, iters=M)` runs the design `M+N` times
(discarding the first `N`) and returns a `BenchmarkResult` with
`avg_us` / `min_us` / `max_us` for both end-to-end host latency and
NPU-only time.  `print_benchmark(result)` formats it nicely.

In [21]:
from aie.utils.benchmark import run_iters, print_benchmark

bench = run_iters(passthrough, in_t, out_t, N=4096, warmup=2, iters=10)
print_benchmark(bench)

NPU time     (avg/min/max us): 174.1 / 164.4 / 187.5
End-to-end   (avg/min/max us): 277.7 / 262.1 / 303.8


## 10. `aie.utils.verify` — sloppy equality for AIE outputs

NPU kernels are often LUT approximations or use saturating arithmetic,
so exact `np.array_equal` is too strict. `count_mismatches(actual, ref,
rtol=0.128, atol=None)` returns `(n_errors, n_checked)` — defaults
to 12.8% relative tolerance and stops at the first inf / nan.
`nearly_equal(...)` returns the boolean mask if you need it.

In [22]:
from aie.utils.verify import count_mismatches, nearly_equal

rng = np.random.default_rng(seed=0)  # deterministic output for the docs
ref = np.linspace(0, 10, 1000, dtype=np.float32)
approx = ref + rng.uniform(-0.05, 0.05, size=ref.shape).astype(np.float32)
errors, n = count_mismatches(approx, ref, rtol=0.05)
print(f"count_mismatches(rtol=0.05): {errors} / {n} samples outside tolerance")

mask = nearly_equal(approx, ref, rtol=0.05)
print(f"nearly_equal: {int(mask.sum())} / {mask.size} samples within tolerance")

count_mismatches(rtol=0.05): 23 / 1000 samples outside tolerance
nearly_equal: 977 / 1000 samples within tolerance


## 11. Putting it all together — full case studies

Single-file `@iron.jit` designs that exercise the patterns above
end-to-end:

| Design | Path |
|--------|------|
| Passthrough kernel + trace | `programming_examples/basic/passthrough_kernel/passthrough_kernel.py` |
| Vector exp (LUT-backed)    | `programming_examples/basic/vector_exp/vector_exp.py` |
| GEMM (parametrized AOT)    | `programming_examples/getting_started/03_matrix_multiplication_single_core/matrix_multiplication_single_core.py` |
| Vector reduce, vector–vector add/mul, vector–scalar mul | `programming_examples/basic/vector_*/` |
| ResNet conv2_x | `programming_examples/ml/resnet/layers_conv2_x/resnet.py` |

The resnet design is the most complete demo of shared-buffer kernel
factories + multi-DMA fan-out: it streams weights to 3 columns at
different sizes and uses both
`conv2dk1_skip_init(skip_input_channels=...)` and
`conv2dk3(weight_output_channels=...)`.

## 12. Deferred follow-ups

A few items surfaced during this walkthrough that are intentionally
**not** part of this PR. Each has a clear scope and a tracking note:

- **`Rtp[T]` annotation for `@iron.jit`** — a first-class runtime scalar
  parameter that gets forwarded into the `aie.runtime_sequence` as a
  non-memref typed block arg, with no recompile per value. Today the
  only way to pass a non-tensor per-call value is `Compile[T]` (which
  recompiles on change) or a 1-element tensor (the
  `vector_scalar_mul` pattern). Guard 1-C in `python/utils/jit.py`
  rejects unannotated scalar params *with defaults* precisely to keep
  this gap from becoming a silent footgun: the default would be baked
  into the kernel and per-call overrides silently ignored.

- **Public `arg_shape(i)` / `arg_dtype(i)` on `BaseKernel`** — replace
  the cryptic `k.arg_types()[i].__args__[0][0]` idiom seen in cells 14
  and 15 with proper public introspection methods. `BaseKernel.tile_size(i)`
  already returns the first dim, but its name is misleading for
  "weights buffer length"-style queries.

- **`programming_guide/mini_tutorial/` migration** — the exercises
  there predate the `In`/`Out` convention and use bare
  `def exercise_N(input0, output)` signatures. They still work
  (positionally) but aren't idiomatic. A separate PR should annotate
  them (~17 files, ~35 params) so Guard 1-C can eventually grow to
  reject *all* unannotated non-`Compile` params, not just the
  silent-default subset.

## ✨ That's the tour!

Every cell here ran on a live Phoenix NPU. The same notebook works
unchanged on Strix (NPU2) — the device auto-detection in §0 handles
both.